In [10]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [11]:
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

True

In [12]:
data = load_faq_data()
index = build_index(data)

In [13]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [14]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
)

response.output_text

"To determine if you can join the course, you'll need to check a few details:\n\n1. **Enrollment Period**: Look for specific enrollment dates and see if the course is still accepting students.\n\n2. **Prerequisites**: Make sure you meet any prerequisites or requirements needed for the course.\n\n3. **Contact Information**: If the details aren't clear, consider reaching out to the course administrator or instructor for confirmation.\n\nIf you provide more context about the course (like its title or institution), I may be able to assist you further!"

In [16]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

### Create  a function tool schema

In [17]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [18]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course?"}', call_id='call_jidyLPa4Te1N60nvSU1bS57N', name='search', type='function_call', id='fc_tmp_6pskn98ys6w', namespace=None, status='completed')]

In [19]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

### Now we send the result back to the LLM

In [20]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [21]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

"Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. \n\nYou don't need to wait for a confirmation email; you're accepted and can start learning and submitting homework right away, even without registering. Just make sure to check the deadlines for submissions!"